# BM Project — Human Data Analysis
**Source:** Liu et al. 2025, Cell Genomics — GSE266330  
**Goal:** Sub-cluster macrophages; identify ER+/SP1-high clusters; characterize by cancer origin, metastasis status, treatment response, gender, tissue site.

In [ ]:
import os, warnings, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
import anndata as ad

warnings.filterwarnings('ignore')
sc.settings.verbosity = 2
sc.settings.n_jobs = 4
%matplotlib inline

OUT     = 'bm_analysis_out'
FIG_DIR = f'{OUT}/figures'
DAT_DIR = f'{OUT}/data'
RAW_DIR = f'{OUT}/raw_geo'
for d in [FIG_DIR, DAT_DIR, RAW_DIR]:
    os.makedirs(d, exist_ok=True)
sc.settings.figdir = FIG_DIR

## 1. Download GSE266330 from GEO

In [ ]:
# Import the helper functions from the script
import sys
sys.path.insert(0, '.')
from bm_macrophage_analysis import (
    download_geo, parse_geo_metadata, load_10x_samples,
    preprocess, annotate_and_extract_macrophages, subcluster_macrophages,
    harmonise_metadata, MACRO_MARKERS, TARGET_GENES
)

extract_dir, meta_gz = download_geo('GSE266330')
print('Extraction dir:', extract_dir)

## 2. Parse clinical metadata

In [ ]:
meta_df = parse_geo_metadata(meta_gz)
print(meta_df.shape)
meta_df.head()

## 3. Load count matrices

In [ ]:
# Load from cache if already preprocessed
full_path = f'{DAT_DIR}/full_preprocessed.h5ad'
if os.path.exists(full_path):
    adata = sc.read_h5ad(full_path)
    print('Loaded from cache:', adata)
else:
    adata = load_10x_samples(extract_dir, meta_df)
    print(adata)

## 4. Pre-processing (full dataset)

In [ ]:
if not os.path.exists(full_path):
    adata = preprocess(adata)
    adata.write_h5ad(full_path)

sc.pl.umap(adata, color='leiden_broad', legend_loc='on data', title='Broad clustering')

## 5. Macrophage isolation

In [ ]:
mac_path = f'{DAT_DIR}/mac_subclustered.h5ad'
if os.path.exists(mac_path):
    mac = sc.read_h5ad(mac_path)
    print('Loaded macrophage subset from cache:', mac)
else:
    mac = annotate_and_extract_macrophages(adata)
    print(mac)

# Verify macrophage markers
markers_present = [g for g in MACRO_MARKERS if g in mac.var_names]
sc.pl.dotplot(mac, var_names=markers_present, groupby='leiden_broad',
              title='Macrophage marker expression in extracted cells')

## 6. Macrophage sub-clustering

In [ ]:
if not os.path.exists(mac_path):
    mac = subcluster_macrophages(mac, resolution=0.4)
    mac.obs = harmonise_metadata(mac.obs)
    mac.write_h5ad(mac_path)

print('Sub-clusters:', mac.obs['mac_cluster'].value_counts())

## 7. UMAP — clusters, cancer type, condition

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

sc.pl.umap(mac, color='mac_cluster', legend_loc='on data',
           title='Macrophage sub-clusters', ax=axes[0, 0], show=False)

ct_col = 'cancer_type' if 'cancer_type' in mac.obs else 'sample_id'
sc.pl.umap(mac, color=ct_col, title='Cancer type', ax=axes[0, 1], show=False)

cond_col = 'condition' if 'condition' in mac.obs else 'mac_cluster'
sc.pl.umap(mac, color=cond_col, title='Condition', ax=axes[1, 0], show=False)

sc.pl.umap(mac, color='sample_id', title='Sample', ax=axes[1, 1], show=False)

fig.suptitle('Macrophage Sub-clustering', fontsize=14)
fig.savefig(f'{FIG_DIR}/01_umap_overview.pdf', bbox_inches='tight')
plt.show()

## 8. Cancer type composition per cluster

In [ ]:
ct_col = 'cancer_type' if 'cancer_type' in mac.obs.columns else None
if ct_col:
    df = (mac.obs.groupby(['mac_cluster', ct_col]).size()
               .reset_index(name='n'))
    df['frac'] = df['n'] / df.groupby('mac_cluster')['n'].transform('sum')
    pivot = df.pivot(index='mac_cluster', columns=ct_col, values='frac').fillna(0)

    fig, ax = plt.subplots(figsize=(max(6, len(pivot)*0.8), 5))
    pivot.plot(kind='bar', stacked=True, ax=ax, colormap='tab20', edgecolor='none')
    ax.set_title('Cancer type per macrophage cluster')
    ax.legend(bbox_to_anchor=(1.01,1), loc='upper left')
    plt.xticks(rotation=0)
    fig.savefig(f'{FIG_DIR}/02_cancer_type_bar.pdf', bbox_inches='tight')
    plt.show()
else:
    print('cancer_type column not found — check metadata harmonisation above')

## 9. ESR1 (ER) and SP1 expression

In [ ]:
adata_plot = mac.raw.to_adata() if mac.raw is not None else mac
genes = [g for g in ['ESR1', 'SP1'] if g in adata_plot.var_names]
print('Genes found:', genes)

if genes:
    fig, axes = plt.subplots(1, len(genes), figsize=(6*len(genes), 5))
    if len(genes) == 1: axes = [axes]
    for ax, g in zip(axes, genes):
        sc.pl.umap(adata_plot, color=g, ax=ax, show=False,
                   color_map='Reds', title=f'{g} expression', vmin=0)
    fig.savefig(f'{FIG_DIR}/03a_feature_plots.pdf', bbox_inches='tight')
    plt.show()

    # Dot plot
    sc.pl.dotplot(mac, var_names=genes, groupby='mac_cluster', use_raw=True,
                  title='ESR1 & SP1 across macrophage sub-clusters',
                  save='03b_dotplot_ESR1_SP1.pdf')

## 10. ER + SP1 co-expression score → identify top cluster

In [ ]:
adata_plot = mac.raw.to_adata() if mac.raw is not None else mac
genes = [g for g in ['ESR1', 'SP1'] if g in adata_plot.var_names]

if len(genes) == 2:
    sc.tl.score_genes(adata_plot, gene_list=genes, score_name='ER_SP1_score')
    mac.obs['ER_SP1_score'] = adata_plot.obs['ER_SP1_score'].values

    top_cl = mac.obs.groupby('mac_cluster')['ER_SP1_score'].mean().idxmax()
    print(f'ER+/SP1-high cluster: {top_cl}')

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    sc.pl.violin(mac, keys='ER_SP1_score', groupby='mac_cluster',
                 ax=axes[0], show=False, rotation=0)
    axes[0].set_title('ER + SP1 score per cluster')
    sc.pl.umap(mac, color='ER_SP1_score', ax=axes[1], show=False,
               color_map='RdYlBu_r', title='ER + SP1 co-expression')
    fig.savefig(f'{FIG_DIR}/07_ER_SP1_coexpression.pdf', bbox_inches='tight')
    plt.show()
    print(mac.obs.groupby('mac_cluster')['ER_SP1_score'].mean().sort_values(ascending=False))

## 11. Metastasis vs. control — cluster enrichment

In [ ]:
cond_col = 'condition' if 'condition' in mac.obs.columns else None
if cond_col and not mac.obs[cond_col].eq('unknown').all():
    df = (mac.obs.groupby([cond_col, 'mac_cluster']).size()
               .reset_index(name='n'))
    df['frac'] = df['n'] / df.groupby(cond_col)['n'].transform('sum')
    pivot = df.pivot(index=cond_col, columns='mac_cluster', values='frac').fillna(0)

    fig, ax = plt.subplots(figsize=(max(6, len(pivot.columns)*0.8), 4))
    pivot.T.plot(kind='bar', ax=ax, edgecolor='none')
    ax.set_title('Macrophage cluster proportion — Metastasis vs. Control')
    ax.legend(title=cond_col, bbox_to_anchor=(1.01,1), loc='upper left')
    plt.xticks(rotation=0)
    fig.savefig(f'{FIG_DIR}/04_metastasis_vs_control.pdf', bbox_inches='tight')
    plt.show()
else:
    print('condition column not available — check GEO metadata')

## 12. DEG per cluster (wilcoxon)

In [ ]:
sc.tl.rank_genes_groups(mac, groupby='mac_cluster', use_raw=True,
                        method='wilcoxon', key_added='rank_mac')
sc.pl.rank_genes_groups(mac, key='rank_mac', n_genes=10,
                        save='deg_per_cluster.pdf')

## 13. Gender + tissue origin

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, col, title in zip(
    axes,
    ['gender', 'tissue_origin'],
    ['Gender per cluster', 'Tissue origin per cluster']
):
    if col not in mac.obs.columns or mac.obs[col].eq('unknown').all():
        ax.text(0.5, 0.5, f"'{col}' not in metadata",
                ha='center', va='center', fontsize=12)
        ax.axis('off'); ax.set_title(title)
        continue
    df = (mac.obs.groupby([col, 'mac_cluster']).size()
               .reset_index(name='n'))
    df['frac'] = df['n'] / df.groupby(col)['n'].transform('sum')
    pivot = df.pivot(index=col, columns='mac_cluster', values='frac').fillna(0)
    pivot.T.plot(kind='bar', ax=ax, edgecolor='none', colormap='Set2')
    ax.set_title(title)
    ax.legend(bbox_to_anchor=(1.01,1), loc='upper left', fontsize=9)
    ax.tick_params(axis='x', rotation=0)

fig.tight_layout()
fig.savefig(f'{FIG_DIR}/06_gender_tissue.pdf', bbox_inches='tight')
plt.show()

## 14. Marker gene summary for ER+/SP1-high cluster
Cross-reference with Yuliang's gene pattern

In [ ]:
# Replace with Yuliang's actual gene list
YULIANG_GENES = ['ESR1', 'SP1', 'CD68', 'MRC1', 'CD163',
                 'TREM2', 'SPP1', 'C1QA', 'APOE', 'FOLR2']

present = [g for g in YULIANG_GENES if g in mac.var_names]
print('Yuliang genes present:', present)

if present:
    sc.pl.dotplot(mac, var_names=present, groupby='mac_cluster',
                  use_raw=True,
                  title="Yuliang's gene signature across macrophage clusters",
                  save='yuliang_signature.pdf')